<a href="https://colab.research.google.com/github/shreyansh6726/LLM-Scratch-pad/blob/main/model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ProtoLLM: A Tiny Transformer for Text Generation

This notebook implements a basic character-level language model based on the Transformer architecture. We will train it on the Tiny Shakespeare dataset to generate text in the style of the famous playwright.

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# --- 1. Hyperparameters ---
batch_size = 32      # How many independent sequences will we process in parallel?
block_size = 64      # What is the maximum context length for predictions?
max_iters = 3000     # Total training steps
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu' # Auto-detect T4 GPU or CPU
n_embd = 128         # Number of embedding dimensions
n_head = 4           # Number of attention heads
n_layer = 4          # Number of transformer blocks
dropout = 0.1
# --------------------------

print(f"Using device: {device}")

Using device: cpu


### Cell 1: Imports and Hyperparameters
**What's happening here?**
- We import `torch` for tensor operations and `nn` for building neural network layers.
- **Hyperparameters** define the scale of our model. `n_embd`, `n_head`, and `n_layer` control the model's capacity, while `batch_size` and `block_size` manage data throughput and context memory.
- We automatically detect if a GPU (`cuda`) is available—essential for training transformers efficiently in Google Colab.

In [2]:
# --- 2. Data Loading & Cleaning ---
# Fetching the tiny shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# Simple Tokenizer building
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # string to list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # list of integers to string

--2026-03-03 14:53:36--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.06s   

2026-03-03 14:53:36 (18.9 MB/s) - ‘input.txt’ saved [1115394/1115394]



### Cell 2: Data Loading and Tokenization
**What's happening here?**
- We download the `input.txt` file containing Shakespeare's works.
- We create a **Character-level Tokenizer**. This maps every unique character in the text to a unique integer (`stoi`) and vice versa (`itos`).
- Unlike word-level tokenizers (like GPT-4's), this model sees every letter and punctuation mark as an individual token.

In [3]:
# Train/Val splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    # Generate a small batch of data of inputs x and targets y
    data_set = train_data if split == 'train' else val_data
    ix = torch.randint(len(data_set) - block_size, (batch_size,))
    x = torch.stack([data_set[i:i+block_size] for i in ix])
    y = torch.stack([data_set[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

### Cell 3: Data Splitting and Batching
**What's happening here?**
- We convert the text into a massive PyTorch tensor and split it: 90% for training, 10% for validation.
- `get_batch` randomly samples `batch_size` chunks of length `block_size` from the data.
- **Important**: In language modeling, the target `y` is simply the input `x` shifted by one position. The model learns to predict the next token at every position.

In [4]:
# --- 3. The Architecture ---

class TransformerBlock(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        # MultiheadAttention provides the core interaction layer
        self.sa = nn.MultiheadAttention(n_embd, n_head, dropout=dropout, batch_first=True)
        self.ffwd = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # 1. Self-Attention + Residual Connection (Pre-norm)
        # We use attn_mask to ensure causality (not looking at future tokens)
        T = x.shape[1]
        mask = torch.triu(torch.ones(T, T, device=x.device) * float('-inf'), diagonal=1)

        normalized_x = self.ln1(x)
        attn_output, _ = self.sa(normalized_x, normalized_x, normalized_x,
                                 attn_mask=mask, need_weights=False)
        x = x + attn_output

        # 2. Feed-Forward + Residual Connection (Pre-norm)
        x = x + self.ffwd(self.ln2(x))
        return x

class ProtoLLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[TransformerBlock(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx) # (B, T, C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T, C)
        x = tok_emb + pos_emb # Add Identity/Positional info

        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x) # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:] # Crop context precisely
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] # focus only on last token
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

### Cell 4: Transformer Architecture
**What's happening here?**
- `TransformerBlock`: This is the "brain" module. It uses **Multi-head Attention** to allow tokens to communicate with each other.
- **Correction**: I added an `attn_mask` (an upper-triangular matrix of negative infinity) to `self.sa(...)`. This ensures that when predicting word $N$, the model cannot "look ahead" at word $N+1$.
- `ProtoLLM`: This combines token embeddings (what the word is) and positional embeddings (where it is in the sentence). It runs the data through multiple blocks before projecting back to characters via `lm_head`.

In [5]:
# --- 4. Training Loop ---
model = ProtoLLM().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

print("Starting training...")
for iter in range(max_iters):
    # Get batch
    xb, yb = get_batch('train')

    # Evaluate loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if iter % 500 == 0:
        print(f"step {iter}: loss {loss.item():.4f}")

Starting training...
step 0: loss 4.3590
step 500: loss 1.8819
step 1000: loss 1.7558
step 1500: loss 1.6369
step 2000: loss 1.4810
step 2500: loss 1.5560


### Cell 5: Training Loop
**What's happening here?**
- We initialize the model on our `device` (GPU).
- We use the **AdamW** optimizer, which is the standard choice for Transformers.
- Every 500 steps, we print the loss. A lower loss means the model is getting better at predicting the next character in Shakespeare's text.

In [7]:
# --- 5. Final Generation ---
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print("\n--- GENERATED TEXT ---")
generated_sequence = model.generate(context, max_new_tokens=500)
print(decode(generated_sequence[0].tolist()))


--- GENERATED TEXT ---

he that mayst I imaging.

Secongar:
Do have more, well deter, thou diviled freele thou
will rend, is, not too have men seem
Our wise toe were that to she sevence
That ourt art both and friar, in ourpocy; an'Forgive
In a maddly neit a lives not, her ady himours;
Movalined arms, it about that you may have slaw
Douls and hear hadred him left.

MRTAUker:
I will see please make to Call from an nothings:
Be keeple thore in earl of kind;
And he would Bariss. I'll preceives out
The well, breed the of wo


### Cell 6: Generation and Output
**What's happening here?**
- We start with a "dummy" token (index 0).
- The model predicts one character, adds it to the sequence, and repeats for 500 steps.
- Finally, we `decode` the integers back into readable text. Note: Early in training, it might look like gibberish, but as loss drops, it starts picking up words and structural patterns like line breaks and names.